# 01 — SG Population Sweep

Generates the SPEC §7.3 deliverables for the SG (`FlexibleGATNet`) architecture across the full matrix:

- **3 regimes:** `kfold-5`, `kfold-10`, `loso`
- **2 mt:** 2, 4
- **3 estimators:** `gnn` (GNNExplainer, primary §5.2), `captum_ig` (CaptumExplainer-IG cross-check §5.3 / §15.7), `attention` (AttentionExplainer cross-check §5.3 / §15.4)

= **18 PopulationResults** total. Each lands at `research/xai/sg/{regime}/mt{N}/{estimator}/` with `node_importance.csv`, `edge_importance.csv`, `channel_pair_matrix.npy`, `feature_importance.csv` (gnn / captum_ig only), `result_meta.json`, `run.json`, and the §8 figures.

> **Runtime note.** GNNExplainer ≈ 1 s/trial at `epochs=200` on GPU; Captum-IG is ~3-5× slower; AttentionExplainer is essentially free. kfold-5 mt2 ≈ 5 folds × ~25 trials = ~3 min for `gnn` / ~10 min for `captum_ig`. **LOSO** = 62 subjects × ~2 trials = ~4 min for `gnn` / ~15 min for `captum_ig` per cell. Run cells one at a time if GPU-bound.

Reference: [`docs/SPEC_xai_graph.md`](../../../docs/SPEC_xai_graph.md) (rev. 4).

In [1]:
%matplotlib inline
import os, sys
from pathlib import Path

import numpy as np
import torch
from scipy import stats

PROJECT_ROOT = Path(os.getcwd()).resolve().parents[2]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.xai import (
    XAIRunConfig,
    run_sg,
    plot_montage_channel_importance, plot_pair_matrix,
    write_run_json,
    PopulationResult,
    CHANNEL_NAMES, N_CH,
)

SG_EXPERIMENT_ROOT = PROJECT_ROOT / "research/experiments/20260506/leak-free-patience-9999/spatial-graph"
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(f"PROJECT_ROOT     = {PROJECT_ROOT}")
print(f"SG_EXPERIMENT_ROOT exists: {SG_EXPERIMENT_ROOT.is_dir()}")
print(f"DEVICE             = {DEVICE}")

PROJECT_ROOT     = /home/user/jeffrymahbuubi/PROJECTS/2-fNIRS-Graph-Base-Method
SG_EXPERIMENT_ROOT exists: True
DEVICE             = cuda:0


## Helper

`run_sg_cell(regime, mt, estimators, gnn_epochs)` runs every estimator for one
`(regime, mt)` matrix cell, persists everything to disk, and returns
`{estimator → PopulationResult}` for the in-notebook summary.

In [2]:
def run_sg_cell(
    regime: str,
    mt: int,
    estimators: tuple = ("gnn", "captum_ig", "attention"),
    gnn_epochs: int = 200,
    gnn_lr: float = 0.01,
) -> dict:
    """Run all `estimators` for one (regime, mt) cell. Persist + plot per estimator."""
    out: dict = {}
    for est in estimators:
        out_dir = PROJECT_ROOT / f"research/xai/sg/{regime}/mt{mt}/{est}"
        out_dir.mkdir(parents=True, exist_ok=True)

        cfg = XAIRunConfig(
            arch="sg", hb="hbo", regime=regime, mt=mt,
            experiment_root=str(SG_EXPERIMENT_ROOT),
            output_dir=str(out_dir),
            device=DEVICE,
            gnn_explainer_epochs=gnn_epochs,
            gnn_explainer_lr=gnn_lr,
            estimator=est,
            seed=42,
        )
        print(f"[{regime} mt={mt}] {est:>10s} → {out_dir.relative_to(PROJECT_ROOT)} ...")
        result = run_sg(cfg)
        result.to_csv(out_dir)
        write_run_json(cfg, result, out_dir)
        plot_montage_channel_importance(result, out_dir)
        plot_pair_matrix(result, out_dir)
        out[est] = result

        top5 = [CHANNEL_NAMES[i] for i in (-result.channel_importance_mean).argsort()[:5]]
        print(f"           done — n_trials={result.n_trials}/{result.n_trials_total} "
              f"({result.included_pct:.1f}%), n_subjects={result.n_subjects}, top-5: {top5}")
    return out


def report_c4_agreement(results: dict, label: str) -> None:
    """SPEC §11 C4: pairwise Spearman ρ between the three estimators' channel rankings."""
    keys = list(results.keys())
    print(f"\n[{label}] C4 channel-rank agreement (top-10 Spearman ρ):")
    for i, a in enumerate(keys):
        for b in keys[i + 1:]:
            ra = (-results[a].channel_importance_mean).argsort().argsort()
            rb = (-results[b].channel_importance_mean).argsort().argsort()
            rho, _ = stats.spearmanr(ra, rb)
            flag = "✓" if rho >= 0.4 else "✗"
            print(f"           ρ({a:>10s}, {b:>10s}) = {rho:+.3f}   {flag} (≥ 0.4)")

## kfold-5 × mt=2

Smallest cell: 5 folds × ~25 val trials each = ~125 trials/estimator.
Total runtime ≈ 5–15 min depending on GPU.

In [ ]:
sg_kfold5_mt2 = run_sg_cell(regime="kfold-5", mt=2)
report_c4_agreement(sg_kfold5_mt2, "kfold-5 × mt=2")

## kfold-5 × mt=4

5 folds × ~50 val trials. ~10–25 min total.

In [ ]:
sg_kfold5_mt4 = run_sg_cell(regime="kfold-5", mt=4)
report_c4_agreement(sg_kfold5_mt4, "kfold-5 × mt=4")

## kfold-10 × mt=2

10 folds × ~12-13 val trials = ~125 trials/estimator. Similar to kfold-5 mt2 in total.

In [ ]:
sg_kfold10_mt2 = run_sg_cell(regime="kfold-10", mt=2)
report_c4_agreement(sg_kfold10_mt2, "kfold-10 × mt=2")

## kfold-10 × mt=4

10 folds × ~25 val trials = ~250 trials/estimator.

In [ ]:
sg_kfold10_mt4 = run_sg_cell(regime="kfold-10", mt=4)
report_c4_agreement(sg_kfold10_mt4, "kfold-10 × mt=4")

## LOSO × mt=2

⚠️  **Long-running.** 62 subjects × ~2 val trials = ~124 trials/estimator. Captum-IG dominates wall time (~30–60 min total). The HbO mt2 LOSO checkpoints are local; HbO mt4 LOSO has a `/root/remote-training-setup/` path that auto-rebases at load time.

In [ ]:
sg_loso_mt2 = run_sg_cell(regime="loso", mt=2)
report_c4_agreement(sg_loso_mt2, "loso × mt=2")

## LOSO × mt=4

⚠️  **Longest.** 62 subjects × ~4 val trials = ~248 trials/estimator. ~1–2 h total.

In [ ]:
sg_loso_mt4 = run_sg_cell(regime="loso", mt=4)
report_c4_agreement(sg_loso_mt4, "loso × mt=4")

## Summary

For every (regime, mt, estimator) combination just produced, this cell prints the top-5 channels and a concise mt=2 ↔ mt=4 stability check (SPEC §11 C3: ρ(mt=2 ranking, mt=4 ranking) ≥ 0.5).

In [ ]:
ALL_RUNS = {
    "kfold-5/mt2":  sg_kfold5_mt2,
    "kfold-5/mt4":  sg_kfold5_mt4,
    "kfold-10/mt2": sg_kfold10_mt2,
    "kfold-10/mt4": sg_kfold10_mt4,
    "loso/mt2":     sg_loso_mt2,
    "loso/mt4":     sg_loso_mt4,
}

print("=" * 78)
print("Top-5 channels per (regime, mt, estimator)")
print("=" * 78)
for cell, res in ALL_RUNS.items():
    for est, r in res.items():
        top5 = [CHANNEL_NAMES[i] for i in (-r.channel_importance_mean).argsort()[:5]]
        print(f"  {cell:>12s} | {est:>10s} : {top5}")

print()
print("=" * 78)
print("SPEC §11 C3 — mt=2 vs mt=4 stability per regime/estimator (ρ ≥ 0.5)")
print("=" * 78)
for regime in ("kfold-5", "kfold-10", "loso"):
    mt2 = ALL_RUNS[f"{regime}/mt2"]
    mt4 = ALL_RUNS[f"{regime}/mt4"]
    for est in ("gnn", "captum_ig", "attention"):
        r2 = (-mt2[est].channel_importance_mean).argsort().argsort()
        r4 = (-mt4[est].channel_importance_mean).argsort().argsort()
        rho, _ = stats.spearmanr(r2, r4)
        flag = "✓" if rho >= 0.5 else "✗"
        print(f"  {regime:>9s} {est:>10s}: ρ(mt2, mt4) = {rho:+.3f}   {flag}")

## Done

Outputs are persisted to `research/xai/sg/{regime}/mt{N}/{estimator}/` and ready for `03_cross_arch_comparison.ipynb`. The C4 (3-way agreement) and C3 (mt=2 vs mt=4) acceptance checks are reported above; C2 (across-fold stability) and C6 (biological-prior overlap with `S1_D1, S5_D5, S3_D3, S2_D1, S4_D5, S4_D7`) live in `03_cross_arch_comparison.ipynb`.